# vLLM 本地单卡快速部署 — 一行命令启动大模型 API（入门实战）

**定位**：用 [vLLM](https://github.com/vllm-project/vllm) 在 **单张 NVIDIA GPU** 上拉起 **OpenAI 兼容 HTTP 服务**，适合第一次接触推理部署的同学；不涉及框架源码修改、KV 缓存调优、多机多卡与高阶性能工程。

**核心口诀**：装好 PyTorch + vLLM 后，终端里 **一条命令** 即可启动服务：

```bash
vllm serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000
```

**开源说明**：本 Notebook 为作者原创示例代码与说明，可自由用于学习、教学及发布到 [Gitee](https://gitee.com) 等代码托管平台；请遵守所下载模型权重与第三方依赖库各自的许可证。

---

## 适用显卡与显存（新手照表选模型）

| 项目 | 说明 |
|------|------|
| **显卡** | **NVIDIA GPU**（推荐 Turing / Ampere / Ada / Blackwell 等较新架构）；本 Demo 按 **单卡** 编写。无 NVIDIA GPU 时无法按本文运行 vLLM CUDA 路径（勿与 CPU 推理混淆）。 |
| **显存** | 默认模型 `Qwen/Qwen2.5-0.5B-Instruct`：**建议 ≥ 6GB 显存**（宽松建议 **≥ 8GB** 更稳）。若改 **7B** 级别模型，常见需要 **≥ 16GB**，并可能需 `--max-model-len` 等参数（本入门 Demo 不展开）。 |
| **驱动** | 安装与所用 **PyTorch / vLLM 轮子**匹配的 NVIDIA 驱动；驱动过旧会导致 `CUDA driver version is insufficient` 类错误。 |
| **CUDA** | 使用与 PyTorch 官方索引一致的 CUDA 运行时版本（如 **cu118 / cu121 / cu124**）；**勿**混装多个冲突的 CUDA 工具包到同一虚拟环境。 |
| **Python** | 代码语法兼容 **Python 3.8+**；**注意**：PyPI 上的 `vllm` 各版本对 **Python 小版本**有下限（常见为 **3.9～3.12** 区间），以 `pip install vllm` 时的报错或 [官方安装文档](https://docs.vllm.ai/en/latest/getting_started/installation.html) 为准。 |
| **系统** | **Linux x86_64** 最省心；Windows 用户建议在 WSL2 或独立终端运行 `vllm serve`，Notebook 内客户端代码同样适用。 |

---

## 完整依赖安装命令（终端执行，推荐先建虚拟环境）
> note: 如果你想省略以下繁琐的环境安装过程，可以一键安装带vllm和pytorch的cuda环境，可以运行docker run --gpus all -ti --net=host --pid=host --ipc=host --privileged --env "HF_TOKEN=$HF_TOKEN" --entrypoint=/bin/bash vllm/vllm-openai:latest
```bash
# 0) 进入你的工作目录，创建并激活虚拟环境
python3 -m venv .venv
source .venv/bin/activate
# Windows（cmd/PowerShell）: .venv\Scripts\activate

# 1) 升级 pip，减少解析依赖时的怪异报错
python -m pip install --upgrade pip

# 2) 安装带 CUDA 的 PyTorch（下面示例为 CUDA 12.4 轮子索引，请按 https://pytorch.org 核对本机驱动是否支持）
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 若你更适配 CUDA 11.8，可改用（示例）：
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3) 安装 vLLM 与调用服务用的 OpenAI 官方 Python SDK
pip install "vllm>=0.6.0" "openai>=1.0.0" "httpx>=0.24.0"

# 如果需要一键安装带vllm和pytorch的cuda环境，可以运行docker run --gpus all -ti --net=host --pid=host --ipc=host --privileged --env "HF_TOKEN=$HF_TOKEN" --entrypoint=/bin/bash vllm/vllm-openai:latest
# 4) 运行本 Notebook 需要 Jupyter（任选其一）
pip install jupyter
```

> **提示**：`vllm` 与 `torch` 的版本组合以当时 PyPI 解析结果为准；若安装失败，请根据终端报错调整 Python 版本或 CUDA 索引，并查阅 vLLM 官方文档的「Installation」章节。

---

## 简单使用说明（Gitee / 本地仓库）

1. 克隆或下载包含本文件的仓库，进入目录，按上一节 **完成依赖安装** 并激活虚拟环境。
2. 在本jupyter notebook所在路径下，启动 Jupyter：`jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root`。
3. **方式 A**：在系统终端执行上文 **核心口诀** 中的 `vllm serve ...`，待日志显示服务就绪后，在 Notebook 里只运行 **环境自检** 与 **调用 API** 单元格（无需再运行「Notebook 内启动服务」）。
4. **方式 B**：从上到下依次执行：环境自检 →（可选）pip 安装 → 启动服务 → 调用 API → 用完后 **停止服务** 释放显存。
5. 首次运行会从 Hugging Face 拉取模型，需网络可达或已配置镜像/缓存；下载时间与带宽有关。

---

In [1]:
# 环境快速自检：Python 版本、CUDA 是否对 torch 可见（逐行说明）

import sys  # 导入标准库 sys，用于读取当前解释器版本信息

print("Python 版本:", sys.version)  # 打印版本字符串，确认与 vLLM 要求的下限一致

try:
    import torch  # 尝试导入 PyTorch，若未安装会进入 except 分支
except ImportError:
    print("未检测到 torch：请先安装带 CUDA 的 PyTorch（见上文 pip 命令）。")  # 给出下一步指引
else:
    print("torch 版本:", torch.__version__)  # 显示已安装的 torch 版本号
    print("CUDA 是否可用:", torch.cuda.is_available())  # True 表示当前进程能看到 GPU
    if torch.cuda.is_available():
        print("GPU0 名称:", torch.cuda.get_device_name(0))  # 打印第一块 GPU 的产品名
        print("GPU0 显存(GB, 约):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))  # 粗算总显存便于对照上文建议


Python 版本: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


torch 版本: 2.10.0+cu129
CUDA 是否可用: True
GPU0 名称: NVIDIA H100 80GB HBM3
GPU0 显存(GB, 约): 79.18


In [2]:
# 可选：在 Notebook 当前内核环境中安装 vLLM 与客户端依赖（若终端已装好可跳过）
# 每一行末尾注释说明该行作用

import subprocess  # 导入 subprocess，用子进程调用 pip，避免手写 shell 字符串
import sys  # 导入 sys，拿到当前 Jupyter 内核对应的 python 可执行路径

_packages = ["vllm>=0.6.0", "openai>=1.0.0", "httpx>=0.24.0"]  # 固定要装的包及最低版本约束
_cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "pip"] + _packages  # 组装：对当前解释器执行 pip install
subprocess.run(_cmd, check=True)  # 运行安装，失败则抛异常便于立刻看到 pip 报错
print("依赖安装命令已执行完毕；若缺少 torch/cuda，请仍按文档单独安装 PyTorch。")  # 提示 PyTorch 需与显卡匹配


依赖安装命令已执行完毕；若缺少 torch/cuda，请仍按文档单独安装 PyTorch。


## 「一行命令」在终端里等价于下面打印的这一行

下一格代码会在 Notebook 里 **启动后台子进程** 做同样的事；你也可以 **复制打印出来的整行** 到终端单独运行，更符合「一条命令启动服务」的习惯。

---

In [3]:
# 在 Notebook 内启动 vLLM OpenAI 兼容 API（与终端「一行 vllm serve」等价）
# 若你已在终端手动执行 vllm serve，请跳过本格，直接运行后面的「调用 API」

import json  # 用于解析 HTTP 返回的 JSON
import shutil  # 用于在 PATH 中查找 vllm 可执行文件
import subprocess  # 用于创建后台子进程
import sys  # 当前 Python 解释器路径，作为模块启动方式的回退
import time  # 用于轮询等待服务就绪
import urllib.error  # 捕获网络与 HTTP 异常
import urllib.request  # 标准库发 HTTP GET，无需额外安装 requests

HOST = "127.0.0.1"  # 监听地址：仅本机访问，降低误暴露风险
PORT = 8000  # 监听端口，与客户端 base_url 保持一致
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Hugging Face 模型 ID，小显存友好

_vllm_process = None  # 全局子进程句柄占位，启动成功后变为 Popen 实例


def one_line_vllm_serve_command():
    """返回与「一行启动」等价的命令列表，并打印可复制的一行 shell 字符串。"""
    vllm_bin = shutil.which("vllm")  # 在 PATH 中查找 vllm 命令
    if vllm_bin:
        parts = [vllm_bin, "serve", MODEL_ID, "--host", HOST, "--port", str(PORT), "--dtype", "half"]  # half 即 FP16，多数消费级卡上更省显存
    else:
        parts = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_ID, "--host", HOST, "--port", str(PORT), "--dtype", "half"]  # 无 vllm 命令时回退到模块入口
    shell_one_liner = " ".join(parts)  # 拼成一条字符串，便于复制到终端
    print("终端一行启动（可直接复制）:")  # 提示用户这是与文档一致的「一行」
    print(shell_one_liner)  # 打印完整命令
    return parts  # 返回列表供 Popen 使用


def start_vllm_if_not_running():
    """启动子进程并轮询 /v1/models 直到可用或超时。"""
    global _vllm_process  # 声明修改模块级变量以保存进程句柄
    if _vllm_process is not None and _vllm_process.poll() is None:
        print("检测到 vLLM 已在运行，跳过重复启动。")  # 避免多实例占满显存
        return
    cmd = one_line_vllm_serve_command()  # 生成命令并打印一行版
    _vllm_process = subprocess.Popen(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )  # 后台运行；日志丢弃避免管道阻塞（排错请到终端手动运行同一命令）
    base = f"http://{HOST}:{PORT}"  # 服务根 URL
    models_url = f"{base}/v1/models"  # OpenAI 兼容模型列表，用于探活
    deadline = time.time() + 600  # 最长等待 600 秒，给首次下载模型留时间
    while time.time() < deadline:
        if _vllm_process.poll() is not None:
            raise RuntimeError(
                "vLLM 进程已退出，退出码 %s。请把上面打印的一行命令复制到终端查看完整日志。" % _vllm_process.returncode
            )  # 子进程挂了立刻报错，避免无限等待
        try:
            with urllib.request.urlopen(models_url, timeout=2) as resp:
                body = resp.read().decode("utf-8")  # 读取响应文本
            data = json.loads(body)  # 解析为字典
            print("服务已就绪，/v1/models 顶层键:", list(data.keys()))  # 简单确认接口可用
            print("OpenAI 兼容 Base URL:", f"http://{HOST}:{PORT}/v1")  # 客户端填写的根路径
            return
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, json.JSONDecodeError):
            time.sleep(2)  # 未就绪则等待 2 秒再试
    raise TimeoutError("等待 vLLM 就绪超时，请检查网络、显存与防火墙。")  # 超时给出明确异常类型


start_vllm_if_not_running()  # 执行启动流程


终端一行启动（可直接复制）:
/usr/local/bin/vllm serve Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 8000 --dtype half


服务已就绪，/v1/models 顶层键: ['object', 'data']
OpenAI 兼容 Base URL: http://127.0.0.1:8000/v1


In [4]:
# 调用本地 vLLM 提供的 OpenAI 兼容 Chat Completions（逐行注释）

from openai import OpenAI  # 导入 OpenAI 官方 SDK 的客户端类（1.x 风格）

HOST = "127.0.0.1"  # 须与启动服务时 --host 一致
PORT = 8000  # 须与启动服务时 --port 一致
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # 须与 serve 时模型名一致

client = OpenAI(
    base_url=f"http://{HOST}:{PORT}/v1",
    api_key="EMPTY",
)  # base_url 指向 vLLM 的 /v1；本地默认不校验密钥，常用占位字符串即可

completion = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "你是帮助新手的简短中文助手。"},
        {"role": "user", "content": "用一句话说明 vLLM 是做什么的。"},
    ],
    temperature=0.2,
    max_tokens=128,
)  # 发起非流式聊天补全请求

text = completion.choices[0].message.content  # 取出第一条回复的正文
print("模型回复:", text)  # 打印模型输出
print("token 用量:", completion.usage)  # 打印 prompt/completion token 统计


模型回复: VLLM是基于深度学习的语言模型，用于生成、理解和解释人类语言。
token 用量: CompletionUsage(completion_tokens=19, prompt_tokens=32, total_tokens=51, completion_tokens_details=None, prompt_tokens_details=None)


In [5]:
# 停止 Notebook 内启动的 vLLM 子进程，释放 GPU 显存（逐行注释）

import os  # 导入 os，用于向进程组发信号（Linux 行为）
import signal  # 导入 signal 模块，使用 SIGTERM 优雅退出

try:
    _vllm_process  # 若未运行过启动格，这里会 NameError，由 except 捕获
except NameError:
    print("未找到 _vllm_process：若服务在终端启动，请在终端 Ctrl+C 结束。")  # 提示用户手动停服务
else:
    if _vllm_process is None or _vllm_process.poll() is not None:
        print("没有正在运行的 Notebook 子进程。")  # 进程已退出或从未启动
    else:
        try:
            os.killpg(os.getpgid(_vllm_process.pid), signal.SIGTERM)  # 对整个进程组发 SIGTERM
        except ProcessLookupError:
            _vllm_process.terminate()  # 进程组接口失败则直接 terminate 子进程
        _vllm_process.wait(timeout=30)  # 等待最多 30 秒释放资源
        print("已请求停止 vLLM 子进程。")  # 告知用户停止流程已走


已请求停止 vLLM 子进程。


## 常见报错与排查（社区速查）

### 1）`torch.cuda.OutOfMemoryError` / 日志中出现 CUDA out of memory

**原因**：当前模型或上下文长度超出显存。**思路**：换更小模型（本 Demo 已用 0.5B）、关闭其它占显存程序、在启动命令中追加更小上下文（例如 `--max-model-len 2048`，具体以 vLLM 版本帮助为准），或换官方支持的量化权重（入门阶段建议先不换参，优先换小模型）。

### 2）`RuntimeError: CUDA driver version is insufficient` 或 PyTorch 无法 `torch.cuda.is_available()`

**原因**：NVIDIA **驱动过旧**，不满足当前 PyTorch/vLLM 预编译包所依赖的 CUDA 用户态版本。**思路**：升级主机显卡驱动到厂商推荐版本；或卸载当前 torch 后，按 [PyTorch 官网](https://pytorch.org) 选择与你驱动匹配的 CUDA 索引 **重装 torch**，再 `pip install -U vllm`。

---

**结语**：掌握「安装 PyTorch + vLLM → 一行 `vllm serve` → 用 OpenAI 兼容客户端调用」后，即可在此基础上再学习 SGLang、批大小、量化与多卡扩展等进阶主题。
